In [127]:
from CADA.paths import DATA_DIRECTORY
import mygene
import collections
import os
import pandas as pd
import re
import pickle

with open(os.path.join(DATA_DIRECTORY, 'processed', 'ids', 'hpo_old_new.dict'), 'rb') as handle:
    hpo_dict = pickle.load(handle)

with open(os.path.join(DATA_DIRECTORY, 'processed', 'ids', 'hpo_id_name.dict'), 'rb') as handle:
    hpo_id_name = pickle.load(handle)

with open(os.path.join(DATA_DIRECTORY, 'processed', 'ids', 'gene_name_id.dict'), 'rb') as handle:
    gene_name_id = pickle.load(handle)



submission_summary = os.path.join(DATA_DIRECTORY, 'raw', 'patients', 'clinvar', 'submission_summary.txt')


clinvar_data = collections.defaultdict(dict)
clinvar_data_vis = collections.defaultdict(dict)

with open(submission_summary, 'r') as infile:
        content = infile.read().splitlines()[16:]
        content = [x.split('\t') for x in content]
        for line in content:
            clinvar_patient = line[10].split('.')[0]
            submitter= line[9]
            gene = line[11]
            significance = line[1]
            phenos_line = line[4]
            omim_id = 'unknown'
            if significance in ['Pathogenic', 'Likely pathogenic']:
                if gene != '-' and gene in gene_name_id:
                    if ';' in phenos_line:
                        temp = phenos_line.split(';')
                        for i in temp:
                            if "OMIM:" in i:
                                omim_id = i
                        
                    clinvar_data[clinvar_patient]['omim'] = omim_id
                    clinvar_data[clinvar_patient]['gene'] = gene_name_id[gene]
                    clinvar_data[clinvar_patient]['submitter'] = submitter
                    clinvar_data[clinvar_patient]['from'] = 'clinvar'
                               
                    clinvar_data_vis[clinvar_patient]['omim'] = omim_id
                    clinvar_data_vis[clinvar_patient]['gene'] = gene
                    clinvar_data_vis[clinvar_patient]['submitter'] = submitter
                    clinvar_data_vis[clinvar_patient]['from'] = 'clinvar'
                               
                    
                        
                        
df_submission = pd.DataFrame.from_dict(clinvar_data, orient='index')
df_submission_vis = pd.DataFrame.from_dict(clinvar_data_vis, orient='index')
df_submission

# df_filtered_identical = df_submission.drop_duplicates()
# df_filtered_identical.to_excel(filtered_out_submission)

#             pheno_line = line[4] 
#             if significance in ['Pathogenic', 'Likely pathogenic']:
#                     if gene != '-1':
#                         if "Human Phenotype Ontology:" in pheno_line:
#                             phenos = re.split(',|;',pheno_line)
#                             hpo_ids = []
#                             omim_id = ''
#                             for item in phenos:
#                                 if 'HP:' in item:
#                                     hpo_id = item.split('y:')[1]
#                                     hpo_ids.append(hpo_id)
#                                 if 'OMIM:' in item:
#                                     omim_id = item.split(':')[1]
#                             if clinvar_patient not in clinvar_data:
#                                 if hpo_ids != []:
#                                     clinvar_data[clinvar_patient]['gene'] = []
#                                     clinvar_data[clinvar_patient]['gene'].append(gene)
#                                     clinvar_data[clinvar_patient]['hpos'] = hpo_ids
#                                     clinvar_data[clinvar_patient]['omim'] = omim_id
#                                     if omim_id not in omims and omim_id != '':
#                                         omims.append(omim_id)

# print('number of disease: ' + str(len(omims)))
# print('number of RCV: ' + str(len(clinvar_data)))

,omim,gene,submitter,from
SCV000020155,unknown,Entrez:9907,OMIM,clinvar
SCV000020156,unknown,Entrez:9907,OMIM,clinvar
SCV000680696,unknown,Entrez:55572,GeneDx,clinvar
SCV000020158,unknown,Entrez:55572,OMIM,clinvar
SCV000020159,unknown,Entrez:55572,OMIM,clinvar
...,...,...,...,...
SCV001161693,unknown,Entrez:5833,OMIM,clinvar
SCV001161694,unknown,Entrez:5833,OMIM,clinvar
SCV001161695,unknown,Entrez:5833,OMIM,clinvar
SCV001161696,unknown,Entrez:85465,OMIM,clinvar


In [148]:
full_release = os.path.join(DATA_DIRECTORY, 'raw', 'patients', 'clinvar', 'scv.lines')
clinvar_data = collections.defaultdict(dict)
clinvar_data_vis = collections.defaultdict(dict)

out_tsv = os.path.join(DATA_DIRECTORY, 'raw', 'patients', 'clinvar', 'clinvar_submissions.tsv')
out_xlsx = os.path.join(DATA_DIRECTORY, 'raw', 'patients', 'clinvar', 'clinvar_submissions.xlsx')

with open(full_release, 'r') as infile:
        content = infile.read().split('ClinVarAccession')
        for i in content[1:]:
            if 'SCV' in i and 'HP:' in i:
                matchacc = re.match(' Acc="(SCV\d+)"', i)
                clinvar_patient = matchacc.group(1)
                hpo_ids = []
                hpo_names = []
                matchhpo = re.finditer('(HP:\d+)', i)
                for hpo in matchhpo:
                    hpo_ids.append(hpo_dict.get(hpo.group(), hpo.group()))
                    hpo_names.append(hpo_id_name[hpo_dict.get(hpo.group(), hpo.group())])
                if len(hpo_ids) > 0 :
                    clinvar_data[clinvar_patient]['hpos'] = ','.join(hpo_ids)
                    clinvar_data_vis[clinvar_patient]['hpos'] = ', '.join(hpo_names)

df_full = pd.DataFrame.from_dict(clinvar_data, orient='index')
df_full_vis = pd.DataFrame.from_dict(clinvar_data_vis, orient='index')
df_full

,hpos
SCV000045941,HP:0001627
SCV000077558,"HP:0000707,HP:0001999,HP:0001999,HP:0001627,HP..."
SCV000077566,"HP:0002311,HP:0008619,HP:0002474"
SCV000077569,HP:0001250
SCV000077571,HP:0001263
...,...
SCV001146969,"HP:0000496,HP:0000508,HP:0001260,HP:0001288,HP..."
SCV001150317,HP:0030127
SCV001150318,"HP:0030127,HP:0008209"
SCV001156159,HP:0011961


In [149]:
df = pd.merge(df_full, df_submission, how = 'inner', left_index=True, right_index=True)
df_vis = pd.merge(df_full_vis, df_submission_vis, how = 'inner', left_index=True, right_index=True)
df_filtered_identical = df.drop_duplicates()
df_filtered_identical_vis = df_vis.drop_duplicates()

In [150]:
df_filtered_identical['patient_id'] = df_filtered_identical.index
df_filtered_identical_vis['patient_id'] = df_filtered_identical_vis.index
df_filtered_identical = df_filtered_identical[['patient_id','omim','gene','hpos','submitter', 'from']]
df_filtered_identical_vis = df_filtered_identical_vis[['patient_id','omim','gene','hpos','submitter', 'from']]
df_filtered_identical_vis = df_filtered_identical_vis.sort_values('gene')

/home/peng/.local/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.
/home/peng/.local/lib/python3.7/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


In [151]:
df_filtered_identical.to_csv(out_tsv, index=False, sep='\t' )
df_filtered_identical_vis.to_excel(out_xlsx, index=None)

In [152]:
df_filtered_identical.to_csv(out_tsv, index=False, sep='\t' )
df_filtered_identical_vis.to_excel(out_xlsx, index=None)